# Undo/Redo շարժիչ

*1991 Academy · Առաքելություններ*

Undo/redo-ն դասագրքային ստեկերի խնդիրն է, թաքնված ամեն իրական հավելվածի ներսում։ Կառուցի՛ր փոքրիկ տեքստային խմբագրիչի վիճակի շարժիչը։

`create_editor()`-ը վերադարձնում է խմբագրիչ՝ `type(text)` (տեքստ ավելացնել), `undo()`, `redo()` և `get_text()` մեթոդներով։ Դասական կանոններ. undo-ն հետ է շրջում վերջին `type`-ը, redo-ն կրկին կիրառում վերջին undo-ն, և նոր բան մուտքագրելը **մաքրում է redo պատմությունը** — ճիշտ ինչպես անում է քո խմբագրիչը։

> 💾 **Աշխատանքդ պահվում է.** Colab-ում վերբեռնված (**File → Upload notebook**) այս նոթբուքը գտնվում է քո Google Drive-ում՝ *Colab Notebooks* թղթապանակում։ Կարող ես փակել այն և շարունակել ավելի ուշ։
>
> ✅ Երբ բոլոր թեստերն անցնեն, վերադարձի՛ր 1991 Academy և սեղմի՛ր **Ես լուծեցի ✓**։

## Քո կոդը

Գրի՛ր լուծումդ ներքևի բջջում և գործարկի՛ր այն. սեղմի՛ր ▶ կամ Shift+Enter։

In [ ]:
# Two stacks: one of past states (for undo),
# one of undone states (for redo).
class Editor:
    def __init__(self):
        pass

    def type(self, text):
        pass  # append text

    def undo(self):
        pass  # revert the last type()

    def redo(self):
        pass  # re-apply the last undone type()

    def get_text(self):
        return ""


def create_editor():
    return Editor()

## Ստուգի՛ր լուծումդ

Գործարկի՛ր ներքևի բջիջը։ Եթե թեստը չանցնի, ուղղի՛ր կոդդ, նորից գործարկի՛ր կոդի բջիջը, հետո՝ այս մեկը։

In [ ]:
#@title ▶ Գործարկել թեստերը { display-mode: "form" }
import json, math, traceback
MSG = {
 "all_pass": "🎉 Բոլոր {n} թեստերն անցան։ Վերադարձի՛ր 1991 Academy և սեղմի՛ր «Ես լուծեցի ✓»։",
 "some_pass": "{p}/{n} թեստ է անցնում։ Շարունակի՛ր։",
 "none_ran": "Ոչ մի թեստ չաշխատեց։ Ֆունկցիայիդ անունը {fn} է՞։",
 "expected": "սպասվում էր {e}, ստացվեց {a}",
 "error": "💥 Թեստերը կանգ առան սխալով՝",
 "not_run": "Գործարկե՞լ ես վերևի կոդի բջիջը։ Թեստերը տեսնում են միայն գործարկված կոդը։"
}

def _report(results, error, fn):
    for name, ok, expected, actual in results:
        print(("✓ " if ok else "✗ ") + name + ("" if ok else "   " + MSG["expected"].format(e=expected, a=actual)))
    if error:
        print("\n" + MSG["error"])
        print(error.rstrip())
    passed = sum(1 for r in results if r[1])
    print()
    if results and passed == len(results) and not error:
        print(MSG["all_pass"].format(n=passed))
    elif results:
        print(MSG["some_pass"].format(p=passed, n=len(results)))
    elif not error:
        print(MSG["none_ran"].format(fn=fn))

def _academy_run_tests():
    results = []

    def __check(name, actual, expected):
        try:
            ok = bool(actual == expected)
        except Exception:
            ok = False
        results.append((str(name), ok, repr(expected), repr(actual)))

    def tests():
        ed = create_editor()
        ed.type("Hello")
        ed.type(" world")
        __check("typing appends", ed.get_text(), "Hello world")
        ed.undo()
        __check("undo reverts last type", ed.get_text(), "Hello")
        ed.redo()
        __check("redo restores it", ed.get_text(), "Hello world")
        ed.undo(); ed.undo()
        __check("undo twice reaches empty", ed.get_text(), "")
        ed.undo()
        __check("undo on empty is safe", ed.get_text(), "")
        ed.redo(); ed.redo()
        __check("redo twice restores both", ed.get_text(), "Hello world")
        ed2 = create_editor()
        ed2.type("a"); ed2.type("b"); ed2.undo(); ed2.type("c"); ed2.redo()
        __check("new typing clears redo", ed2.get_text(), "ac")
        ed3 = create_editor()
        ed3.type("x"); ed3.undo()
        ed4 = create_editor()
        ed4.redo()
        __check("each editor has its own history", ed4.get_text(), "")

    error = None
    try:
        tests()
    except Exception as exc:
        error = traceback.format_exc(limit=-3)
        if isinstance(exc, NameError):
            error += "\n" + MSG["not_run"]
    _report(results, error, 'create_editor')

_academy_run_tests()

## Հուշումներ

<details><summary>Հուշում 1</summary>

Պահի՛ր ամբողջ տեքստի snapshot-ների պատմության ստեկ։ type()-ը փոփոխելուց առաջ push է անում ընթացիկ տեքստը. undo()-ն ընթացիկ տեքստը push է անում redo ստեկ և pop անում պատմությունից։

</details>

<details><summary>Հուշում 2</summary>

type()-ը պետք է դատարկի redo ստեկը — դա «ճյուղավորվող պատմության» կանոնն է, որին հետևում են խմբագրիչները։

</details>

<details><summary>Հուշում 3</summary>

Ստեկերը պահի՛ր self-ի վրա (self.history = [] __init__-ում), ոչ թե դասի մակարդակի փոփոխականներում. դրանք կկիսվեին բոլոր խմբագրիչների միջև։

</details>